In [0]:
%run ../00_common/data_utils

In [0]:
def update_ukey_manually(csv_path, master_table):
    """
    从 CSV 读取 MarketCode, BrandCode, SourceSystemCode, ConsumerId, CONSUMERMDMKey，
    匹配 t_master_consumer 的四个 business key，更新 consumermdmkey 字段。
    """
    # 1. 读取 CSV，统一 cast 为 string 避免类型不匹配
    csv_df = (
        spark.read
        .option("header", "true")
        .csv(csv_path)
        .select(
            F.col("MarketCode").cast("string"),
            F.col("BrandCode").cast("string"),
            F.col("SourceSystemCode").cast("string"),
            F.col("ConsumerId").cast("string"),
            F.col("CONSUMERMDMKey").cast("string"),
        )
    )

    # 过滤任意字段为空(null或空字符串)的行
    csv_df = csv_df.filter(
        (F.trim(F.col("MarketCode")) != "") &
        (F.trim(F.col("BrandCode")) != "") &
        (F.trim(F.col("SourceSystemCode")) != "") &
        (F.trim(F.col("ConsumerId")) != "") &
        (F.trim(F.col("CONSUMERMDMKey")) != "")
    )

    csv_count = csv_df.count()

    # 按 business key 去重，避免 merge 时匹配到多条报错
    bk_cols = ["MarketCode", "BrandCode", "SourceSystemCode", "ConsumerId"]
    csv_df = csv_df.dropDuplicates(bk_cols)
    dedup_count = csv_df.count()
    if csv_count != dedup_count:
        print(f"CSV rows: {csv_count}, after dedup by business key: {dedup_count} (dropped {csv_count - dedup_count})")
    else:
        print(f"CSV rows: {csv_count}")

    # 2. left join 匹配 master 记录，区分 matched / unmatched
    joined_df = (
        csv_df.alias("c")
        .join(spark.table(master_table).alias("m"),
            (F.col("c.MarketCode") == F.col("m.scon_mrkt_code")) &
            (F.col("c.BrandCode") == F.col("m.scon_brnd_code")) &
            (F.col("c.SourceSystemCode") == F.col("m.scon_srcs_code")) &
            (F.col("c.ConsumerId") == F.col("m.scon_consumerid")),
            "left")
        .select(
            F.col("m.scon_mrkt_code"),
            F.col("m.scon_brnd_code"),
            F.col("m.scon_srcs_code"),
            F.col("m.scon_consumerid"),
            F.col("m.consumermdmkey").alias("old_consumermdmkey"),
            F.col("c.CONSUMERMDMKey").alias("new_consumermdmkey"),
        )
    )

    # 日志：CSV 中未匹配到的行（用 master 的 business key 判断，consumermdmkey 可能在 master 中为 null）
    unmatched_df = joined_df.filter(F.col("scon_consumerid").isNull())
    unmatched_count = unmatched_df.count()
    if unmatched_count > 0:
        print(f"WARNING: {unmatched_count} CSV rows not found in t_master_consumer:")
        unmatched_df.show(unmatched_count, truncate=False)

    # 匹配上的，过滤掉新旧 ukey 相同的（无需更新），old ukey 为 null 的也要更新
    update_df = (
        joined_df
        .filter(F.col("scon_consumerid").isNotNull())
        .filter((F.col("old_consumermdmkey").isNull()) | (F.col("old_consumermdmkey") != F.col("new_consumermdmkey")))
    )

    # 同一个 BK 匹配到多条 master 记录时，保留一条（避免 merge 报错）
    join_key_cols = ["scon_mrkt_code", "scon_brnd_code", "scon_srcs_code", "scon_consumerid"]
    update_df = update_df.dropDuplicates(join_key_cols)

    update_count = update_df.count()
    print(f"Rows to update: {update_count}")

    if update_count == 0:
        print("Nothing to update.")
        return

    # 3. Delta merge 更新 consumermdmkey
    (
        DeltaTable.forName(spark, master_table).alias("b")
        .merge(update_df.alias("u"), """
            b.scon_mrkt_code = u.scon_mrkt_code AND
            b.scon_brnd_code = u.scon_brnd_code AND
            b.scon_srcs_code = u.scon_srcs_code AND
            b.scon_consumerid = u.scon_consumerid
        """)
        .whenMatchedUpdate(set={
            "consumermdmkey": F.col("u.new_consumermdmkey"),
        })
        .execute()
    )

    print(f"Updated {update_count} rows in t_master_consumer.")

In [0]:
csv_path = dbutils.widgets.get("csv_path")
master_table = dbutils.widgets.get("master_table")
print(f"csv_path: {csv_path}")
print(f"master_table: {master_table}")

if not csv_path or csv_path.strip() == "":
    raise ValueError("csv_path is required")
if not master_table or master_table.strip() == "":
    raise ValueError("master_table is required")

update_ukey_manually(csv_path, master_table)